# Multi-Agent Trading Environment: Local Policy Training (135M Model)

This notebook trains a 135M parameter model (**SmolLM-135M**) using Chain-of-Thought (CoT) prompting to step into the role of the `QuantTrader`.

## Environment Setup

In [ ]:
%%capture
# Install Unsloth and dependencies
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps xformers trl peft accelerate bitsandbytes

## 1. Init Model (SmolLM-135M)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/SmolLM-135M-Instruct-bnb-4bit", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 2. Dataset Generation (Trajectory Collection)
Expects real trajectories from `checkpoints/sft_trajectories.jsonl`.

In [ ]:
from datasets import Dataset
import json
import os

SYSTEM_PROMPT = """You are a Quant Trader. Analyze the signals and state, then provide a decision.

State: {state}
Signals: {signals}
"""

trajectory_path = "../checkpoints/sft_trajectories.jsonl"
records = []
if os.path.exists(trajectory_path):
    with open(trajectory_path, "r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            # Filter for high-quality trades/episodes (0.65+ grade)
            if row.get("final_grade", 0.0) >= 0.65:
                records.append({
                    "state": json.dumps(row["state"]),
                    "signals": json.dumps({
                        "ta": row["signals"]["ta_score"],
                        "fa": row["signals"]["fa_sentiment"],
                        "risk": row["signals"]["position_limit"]
                    }),
                    "action": json.dumps(row["action"]),
                    "reasoning": row["signals"]["reasoning"]["trader"] if "reasoning" in row["signals"] else "Market signals suggest this trade action based on combined technical and risk parameters."
                })

if not records:
    # Small synthetic fallback if no data exists
    records = [
        {
            "state": "[1.0, 1.05, 0.98]",
            "signals": "{\"ta\": 0.8, \"fa\": 0.5, \"risk\": 1.0}",
            "action": "{\"direction\": 1, \"size\": 0.68}",
            "reasoning": "Bullish momentum from Quant Researcher combined with strong risk allocation suggests a buy."
        }
    ]

def format_prompts(examples):
    texts = []
    for state, signals, action, reasoning in zip(examples["state"], examples["signals"], examples["action"], examples["reasoning"]):
        prompt = SYSTEM_PROMPT.format(state=state, signals=signals)
        text = f"{prompt}\n### Analysis:\n1. Market Trend: Positive\n2. Risk Level: Controlled\n3. Final Decision:\n{action}{tokenizer.eos_token}"
        texts.append(text)
    return {"text": texts}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_prompts, batched=True)

## 3. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150, # More steps for CoT stability
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 4. Save to `models/local_policy/`

In [ ]:
import os
save_dir = "../models/local_policy"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained_merged(save_dir, tokenizer, save_method = "merged_16bit")